# Indexing Pipeline — Google Colab

Runs the full embedding + Qdrant indexing pipeline on Colab GPU/CPU, then packages the resulting storage folder for download to your local machine.

**Steps:**
1. Install dependencies
2. Start Qdrant binary (localhost:6333 inside this VM)
3. Upload `chunks.json` from your machine
4. Upload `init_vectordB.py` and `index_chunks.py`
5. Initialize collections
6. Run indexing (heavy embedding here, not on your machine)
7. Verify point counts
8. Download storage zip → unzip into your local `data/qdrant_dB/`

## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install qdrant-client fastembed

## Cell 2 — Download and start Qdrant binary

Qdrant provides a pre-built Linux binary. We run it as a background process inside this VM on the standard port 6333.

**Important:**
- Make sure you set the runtime to T4 GPU before running below cell 
- `Runtime → Change runtime type → T4 GPU` 
- FastEmbed auto-detects CUDA — you'll get ~3x speedup over CPU with no code changes.

In [ ]:
import os
import subprocess
import time
import urllib.request

QDRANT_VERSION = "v1.13.2"
QDRANT_STORAGE = "/content/qdrant_storage"
os.makedirs(QDRANT_STORAGE, exist_ok=True)

# Download
binary_url = f"https://github.com/qdrant/qdrant/releases/download/{QDRANT_VERSION}/qdrant-x86_64-unknown-linux-musl.tar.gz"
print(f"Downloading Qdrant {QDRANT_VERSION}...")
urllib.request.urlretrieve(binary_url, "qdrant.tar.gz")
!tar -xzf qdrant.tar.gz
!chmod +x qdrant
print("Binary ready.")

# Set storage path via environment variable instead of CLI flag
env = os.environ.copy()
env["QDRANT__STORAGE__STORAGE_PATH"] = QDRANT_STORAGE

# Start Qdrant
qdrant_process = subprocess.Popen(
    ["./qdrant"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Poll until ready
print("Waiting for Qdrant to start", end="")
for _ in range(60):
    time.sleep(1)
    try:
        urllib.request.urlopen("http://localhost:6333/", timeout=2)
        print(" ready!")
        break
    except Exception:
        print(".", end="", flush=True)
else:
    qdrant_process.terminate()
    raise RuntimeError("Qdrant failed to start within 60s.")

print(f"Qdrant running at http://localhost:6333/")

## Cell 3 — Upload your files

This will open file pickers. Upload in this order:
1. `data/processed/chunks.json`
2. `src/vectordB/init_vectordB.py`
3. `src/vectordB/index_chunks.py`

In [ ]:
from google.colab import files
import shutil

print("=== Upload chunks.json ===")
uploaded = files.upload()  # select chunks.json

# Place it where index_chunks.py expects it
os.makedirs("data/processed", exist_ok=True)
for fname in uploaded:
    shutil.move(fname, f"data/processed/{fname}")
print(f"chunks.json saved to data/processed/")

In [ ]:
print("=== Upload init_vectordB.py and index_chunks.py ===")
uploaded = files.upload()  # select both .py files

os.makedirs("src/vectordB", exist_ok=True)
for fname in uploaded:
    shutil.move(fname, f"src/vectordB/{fname}")
    print(f"  Saved: src/vectordB/{fname}")

## Cell 4 — Verify uploads

In [ ]:
import json

# Verify chunks.json
with open("data/processed/chunks.json") as f:
    chunks = json.load(f)

parents  = [c for c in chunks if c["type"] == "parent"]
children = [c for c in chunks if c["type"] == "child"]
print(f"chunks.json loaded: {len(chunks)} total — {len(parents)} parents, {len(children)} children")

# Verify scripts
for f in ["src/vectordB/init_vectordB.py", "src/vectordB/index_chunks.py"]:
    exists = os.path.exists(f)
    print(f"  {'✅' if exists else '❌'} {f}")

## Cell 5 — Initialize Qdrant collections

In [ ]:
!python src/vectordB/init_vectordB.py

## Cell 6 — Run indexing pipeline

This is the heavy step. BGE-large-en-v1.5 embeds ~27K chunks.

**Expected time:**
- Colab CPU: ~25-35 minutes
- Colab T4 GPU: ~8-12 minutes

FastEmbed auto-detects and uses GPU if available.

In [ ]:
!python src/vectordB/index_chunks.py

## Cell 7 — Verify point counts in both collections

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")

for collection in ["documentation_chunks", "documentation_parents"]:
    info = client.get_collection(collection)
    count = info.points_count
    print(f"{collection}: {count} points")

# Sanity check against chunks.json
print(f"\nExpected — children: {len(children)}, parents: {len(parents)}")

## Cell 8 — Package and download storage

Zips the entire Qdrant storage folder and downloads it.

**After download, on your local machine:**
```bash
# 1. Stop your local Qdrant if running
cd docker && docker compose down

# 2. Clear existing (empty) storage
rm -rf data/qdrant_dB/*

# 3. Unzip the downloaded file into your storage folder
unzip qdrant_storage.zip -d data/qdrant_dB/

# 4. Start Qdrant — it will read the pre-built index instantly
docker compose up -d
```
Verify at http://localhost:6333/dashboard — both collections should show correct point counts immediately, no re-indexing.

In [ ]:
import shutil
from google.colab import files

# Shut down Qdrant cleanly before zipping so all WAL segments are flushed to disk
print("Shutting down Qdrant for clean snapshot...")
qdrant_process.terminate()
qdrant_process.wait()
print("Qdrant stopped.")

# Zip the storage folder
print("Zipping storage...")
shutil.make_archive("/content/qdrant_storage", "zip", QDRANT_STORAGE)
zip_size_mb = os.path.getsize("/content/qdrant_storage.zip") / (1024 * 1024)
print(f"Archive ready: qdrant_storage.zip ({zip_size_mb:.1f} MB)")

# Download to your machine
files.download("/content/qdrant_storage.zip")